In [ ]:
class TransformerFECG_CNN(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_encoder_layers, dim_feedforward, conv_channels):
        super(TransformerFECG_CNN, self).__init__()

        self.encoder_conv1 = nn.Sequential(
            nn.Conv1d(input_dim, conv_channels, kernel_size=3, padding='same'),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU()
        )

        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.encoder_conv2 = nn.Sequential(
            nn.Conv1d(conv_channels, conv_channels, kernel_size=3, padding='same'),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU()
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model,
            nhead,
            dim_feedforward,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )

        self.LSTM = nn.LSTM(d_model, d_model, batch_first = True)

        self.decoder_conv1 = nn.Sequential(
            nn.Conv1d(conv_channels, conv_channels, kernel_size=3, padding='same'),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU()
        )

        self.upsample = nn.Upsample(scale_factor=2, mode='linear', align_corners=False)

        self.decoder_conv2 = nn.Sequential(
            nn.Conv1d(conv_channels, 64, kernel_size=3, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.qrs_head = nn.Sequential(
            nn.Conv1d(64, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, encoder_conv1_input):
        encoder_conv1_input = encoder_conv1_input
        encoder_conv1 = self.encoder_conv1(encoder_conv1_input)
        pool1 = self.pool1(encoder_conv1)
        encoder_conv2 = self.encoder_conv2(pool1)

        encoder_conv2 = encoder_conv2.permute(0, 2, 1)
        transformer_encoder = self.transformer_encoder(encoder_conv2)

        LSTM, _ = self.LSTM(transformer_encoder)
        LSTM = LSTM.permute(0, 2, 1)

        decoder_conv1 = self.decoder_conv1(LSTM)
        upsample = self.upsample(decoder_conv1)
        decoder_conv2 = self.decoder_conv2(upsample)

        qrs_out = self.qrs_head(decoder_conv2)

        return qrs_out

In [ ]:
class Bloco_CNN(nn.Module):
    def __init__(self, dim_ref):
        super(Bloco_CNN, self).__init__()

        self.encoder_conv1 = nn.Sequential(
            nn.Conv1d(dim_ref, 64, kernel_size=3, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.MaxPool1 = nn.MaxPool1d(kernel_size=2)

        self.encoder_conv2 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.LSTM = nn.LSTM(64, 64, batch_first = True)

        self.decoder_conv1 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.UpSample1 = nn.Upsample(scale_factor=2, mode='linear', align_corners=False)

        self.decoder_conv2 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.qrs_head = nn.Sequential(
            nn.Conv1d(64, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, encoder_conv1_input):
        encoder_conv1 = self.encoder_conv1(encoder_conv1_input)
        MaxPool1 = self.MaxPool1(encoder_conv1)
        encoder_conv2 = self.encoder_conv2(MaxPool1)

        encoder_conv2 = encoder_conv2.permute(0, 2, 1)
        LSTM, _ = self.LSTM(encoder_conv2)
        LSTM = LSTM.permute(0, 2, 1)

        decoder_conv1 = self.decoder_conv1(LSTM)
        UpSample1 = self.UpSample1(decoder_conv1)
        decoder_conv2 = self.decoder_conv2(UpSample1)
        qrs_out = self.qrs_head(decoder_conv2)

        return qrs_out

In [ ]:
class Refinamento_CNN(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_encoder_layers, dim_feedforward, conv_channels, dim_ref):
        super(Refinamento_CNN, self).__init__()

        self.TransformerFECG_CNN = TransformerFECG_CNN(input_dim, d_model, nhead, num_encoder_layers, dim_feedforward, conv_channels)
        self.norm = nn.BatchNorm1d(dim_ref)
        self.Bloco_CNN = Bloco_CNN(dim_ref)

    def forward(self, TransformerFECG_input):
        qrs_out = self.TransformerFECG_CNN(TransformerFECG_input)
        concat = torch.cat((TransformerFECG_input, qrs_out), dim=1)
        in_ref = self.norm(concat)
        qrs_refinado = self.Bloco_CNN(in_ref)
        return qrs_out, qrs_refinado

input_dim = 4

dim_ref = input_dim + 1

d_model = 128

nhead = 1

conv_channels = 128

num_encoder_layers = 1

dim_feedforward = 256